# TrustRAG C3 - type head

Thin by design: this notebook clones, installs and calls `src/`. Every line of
logic lives in the repo so the run can be reproduced without a notebook.

Needs a GPU accelerator (P100 or T4 x2) and internet ON for the clone and the
model download. Roughly one to two hours.

It trains ONLY the C3 type head. It does not touch C1's checkpoint, C1's
splits or anything under `results/c1/` - those are frozen and every C2 number
assumes them unchanged.

In [ ]:
import os, subprocess, sys

REPO = "/kaggle/working/trustrag"
if not os.path.isdir(REPO):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Wimukthi316/TrustRAG.git", REPO],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
%pip install -q "transformers>=4.48" datasets accelerate scikit-learn pyyaml

## Inputs

The type head trains on gold spans from the frozen processed corpus and is
evaluated on both gold spans and the typed predicted spans Layer 2 produced.
Attach the dataset that carries `data/processed/`, `results/c1/` and
`results/c3/analysis/` and point these paths at it, or upload them.

The cell below fails loudly rather than training on whatever it can find.

In [ ]:
from pathlib import Path

REQUIRED = [
    "data/processed/ragtruth_train.jsonl",
    "data/processed/ragtruth_test.jsonl",
    "results/c1/modernbert-base/split_ids.json",
    "results/c3/analysis/typed_predictions_test.jsonl",
    "results/c3/analysis/typed_predictions_calib.jsonl",
]
missing = [p for p in REQUIRED if not Path(p).exists()]
if missing:
    raise SystemExit("missing inputs:\n  " + "\n  ".join(missing))
print("all inputs present")

In [ ]:
# Smoke test first: 64 spans, one epoch. If this fails, the full run would too,
# an hour later.
!python -m src.c3_explanation.train_c3_type --config configs/c3_type.yaml --limit 64 --epochs 1

In [ ]:
!python -m src.c3_explanation.train_c3_type --config configs/c3_type.yaml

In [ ]:
# Writes results/c3/type/head_probs_*.jsonl in exactly the shape
# type_conformal.py already reads, so Layer 4 runs on the head with no change.
!python -m src.c3_explanation.evaluate_c3_type --config configs/c3_type.yaml

In [ ]:
!python -m src.c3_explanation.type_conformal \
    --calib results/c3/type/head_probs_predicted_calib.jsonl \
    --test results/c3/type/head_probs_predicted_test.jsonl \
    --out-name c3_conformal_head.json

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/c3_type_artifacts", "zip", "results/c3")
print("download /kaggle/working/c3_type_artifacts.zip")